# 시간 관련 컬럼 파싱 실패한 8개 파일 추적

In [1]:
import pandas as pd
from pathlib import Path

# 1. 메타데이터 엑셀에서 실패한 파일 목록 추출
excel_path = '../metadata/sdot-schema-mapping.xlsx' # 엑셀 파일명 확인 필요
raw_dir = Path('../data/raw')

df_inv = pd.read_excel(excel_path, sheet_name='FileInventory')
failed_files = df_inv[df_inv['remarks'] != '성공'].copy()

print(f"🚨 디버깅 대상 파일: {len(failed_files)}개\n")

# 2. 각 파일의 타겟 컬럼 샘플 데이터 확인
for _, row in failed_files.iterrows():
    file_path = raw_dir / row['relative_path']
    target_col = row['target_time_col']
    
    print(f"📄 파일명: {row['file_name']}")
    print(f"   - 타겟 컬럼: {target_col} | 스키마: {row['schema_version']}")
    
    try:
        # 인코딩 맞춰서 읽기 (기존 스크립트처럼 index_col=False 유지)
        df = pd.read_csv(file_path, encoding=row['encoding'], nrows=5, index_col=False)
        
        if target_col in df.columns:
            samples = df[target_col].tolist()
            print(f"   - 데이터 샘플: {samples}\n")
        else:
            print(f"   - ⚠️ {target_col} 컬럼이 실제로 존재하지 않음!\n")
            
    except Exception as e:
        print(f"   - 🚨 읽기 에러: {e}\n")

🚨 디버깅 대상 파일: 8개

📄 파일명: S-DoT_NATURE_2021.01.25-01.31.csv
   - 타겟 컬럼: 전송시간 | 스키마: v2020_2022_corrupted
   - 데이터 샘플: [20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0]

📄 파일명: S-DoT_NATURE_2021.02.22-02.28.csv
   - 타겟 컬럼: 전송시간 | 스키마: v2020_2022_corrupted
   - 데이터 샘플: [20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0]

📄 파일명: S-DoT_NATURE_2021.04.12-04.18.csv
   - 타겟 컬럼: 전송시간 | 스키마: v2020_2022_corrupted
   - 데이터 샘플: [20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0]

📄 파일명: S-DoT_NATURE_2021.04.19-04.25.csv
   - 타겟 컬럼: 전송시간 | 스키마: v2020_2022_corrupted
   - 데이터 샘플: [20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0]

📄 파일명: S-DoT_NATURE_2021.04.26-05.02.csv
   - 타겟 컬럼: 전송시간 | 스키마: v2020_2022_corrupted
   - 데이터 샘플: [20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0, 20200000000000.0]

📄 파일명: S-DoT_NATURE_2021.09.

In [2]:
import re

def extract_dates_from_filename(filename):
    """파일명에서 2021.01.25-01.31 형태를 추출하여 시작/종료일로 변환"""
    # 정규식 패턴: 연도(4자리).월(2자리).일(2자리)-월(2자리).일(2자리)
    pattern = r'(\d{4})\.(\d{2})\.(\d{2})-(\d{2})\.(\d{2})'
    match = re.search(pattern, filename)
    
    if match:
        year, start_month, start_day, end_month, end_day = match.groups()
        
        # 시작일은 00:00:00, 종료일은 23:59:59로 세팅
        start_date = f"{year}-{start_month}-{start_day} 00:00:00"
        
        # 만약 12.28-01.03 처럼 연도를 넘기는 주간이라면? 
        # (S-DoT 데이터 특성상 연말/연초 걸치는 경우 대비)
        end_year = year
        if int(start_month) == 12 and int(end_month) == 1:
            end_year = str(int(year) + 1)
            
        end_date = f"{end_year}-{end_month}-{end_day} 23:59:59"
        
        return start_date, end_date
    return None, None

# 실패했던 파일명들로 테스트
test_filenames = [
    "S-DoT_NATURE_2021.01.25-01.31.csv",
    "S-DoT_NATURE_2021.09.30-10.05.csv",
    "S-DoT_NATURE_2022.12.14-12.18.csv"
]

for fname in test_filenames:
    s_date, e_date = extract_dates_from_filename(fname)
    print(f"파일명: {fname}\n -> 시작: {s_date} | 종료: {e_date}\n")

파일명: S-DoT_NATURE_2021.01.25-01.31.csv
 -> 시작: 2021-01-25 00:00:00 | 종료: 2021-01-31 23:59:59

파일명: S-DoT_NATURE_2021.09.30-10.05.csv
 -> 시작: 2021-09-30 00:00:00 | 종료: 2021-10-05 23:59:59

파일명: S-DoT_NATURE_2022.12.14-12.18.csv
 -> 시작: 2022-12-14 00:00:00 | 종료: 2022-12-18 23:59:59

